# `n_buy` and `n_funds` — signals and performance

Builds two security-quarter signals directly from the raw deliverables and evaluates them.

| signal | what it is |
|---|---|
| `n_funds` | how many funds hold the security that quarter (**coverage**) |
| `n_buy` | how many of them **raised their active weight** that quarter (**buying**) |

Also shown: `frac_buy` (the rate `n_buy/n_funds`) and `n_buy_resid` (`n_buy` with coverage regressed out), because `corr(n_buy, n_funds) ≈ 0.96` — so the two headline signals are nearly the same thing, and the diagnostics reveal whether any performance is *buying* or just *coverage*.

**Inputs** (edit `ROOT` below): `<ROOT>/manager_holdings/panel_holdings_All_Funds_{2002..2024}.parquet` and `<ROOT>/return_data_v2.csv`.

All the logic lives in `signals_perf.py`; this notebook only drives it and plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import signals_perf as S

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

# ROOT is the folder that CONTAINS manager_holdings/ and return_data_v2.csv
ROOT = '.'
cfg = S.Config(root=ROOT)
cfg

In [ ]:
# Load -> build signals -> evaluate. One call does everything.
res = S.run(cfg)

## How related are the two signals?

If `n_buy` is essentially `n_funds`, then any 'buying' result is really a coverage/popularity effect.

In [ ]:
res.corr.round(3)

## Performance table

For each signal × horizon × sample: number of quarters, mean quarterly long-short return, Newey-West *t*, hit rate, annualised return, annualised Sharpe, cumulative return, max drawdown.

`sample`: **discovery** = 2002Q1–2013Q4, **validation** = 2014Q1–2024Q4 (held out), **all** = full sample. `horizon` 1/2/3 = holding t→t+1, t+1→t+2 (predictive), t+2→t+3 (tradeable, clears the ~45–60 day filing delay).

In [ ]:
perf = res.perf
cols = ['signal','horizon','sample','n_quarters','mean_q','t_nw','hit',
        'ann_return','sharpe_ann','cum_return','max_drawdown']
perf[cols].round(4)

In [ ]:
# The headline view: n_buy vs n_funds, full sample, all horizons.
head = perf[(perf.signal.isin(['n_buy','n_funds'])) & (perf['sample']=='all')]
head[['signal','horizon','n_quarters','mean_q','t_nw','sharpe_ann','hit','ann_return']].round(4)

## `t`-stat by signal and horizon (discovery vs validation)

The honest test is whether a discovery-sample *t* survives out of sample.

In [ ]:
tt = perf[perf['sample'].isin(['discovery','validation'])].pivot_table(
        index=['signal','horizon'], columns='sample', values='t_nw')
tt.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
order = [(s,h) for s in ['n_funds','n_buy','frac_buy','n_buy_resid'] for h in (1,2,3)]
labels = [f'{s}\nh{h}' for s,h in order]
allt = perf[perf['sample']=='all'].set_index(['signal','horizon'])['t_nw']
vals = [allt.get((s,h), np.nan) for s,h in order]
colors = ['#4C78A8' if abs(v)>=2 else '#B0B0B0' for v in vals]
ax.bar(range(len(vals)), vals, color=colors)
ax.axhline(0, color='k', lw=0.8); ax.axhline(2, color='r', ls='--', lw=0.8)
ax.axhline(-2, color='r', ls='--', lw=0.8)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Newey-West t (full sample)')
ax.set_title('Long-short t-stat by signal and horizon  (|t|>=2 highlighted)')
plt.tight_layout(); plt.show()

## Cumulative long-short return

Wealth from \$1 compounding the per-quarter top-minus-bottom spread (horizon 1).

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for sig in ['n_buy','n_funds','frac_buy','n_buy_resid']:
    sp = res.spreads[f'{sig}|h1'].sort_index()
    wealth = (1+sp).cumprod()
    ax.plot(sp.index.to_timestamp(), wealth, label=sig, lw=1.6)
ax.axvline(pd.Period('2014Q1','Q').to_timestamp(), color='k', ls=':', lw=1,
           label='discovery | validation')
ax.axhline(1, color='k', lw=0.6)
ax.set_ylabel('growth of $1'); ax.set_title('Cumulative long-short return (horizon 1)')
ax.legend(); plt.tight_layout(); plt.show()

## Reading the result

- `n_buy` and `n_funds` post similar, positive *t* — unsurprising given `corr ≈ 0.96`.
- `frac_buy` (the actual buying **rate**) and `n_buy_resid` (buying with coverage removed) are the controls. If those collapse while `n_buy`/`n_funds` do not, the effect is **institutional coverage/breadth**, not fund buying.
- Compare horizons: a signal that only appears at h1 and dies by h2/h3 is not implementable (the ~45–60 day 13F filing lag lands in that window).

In [ ]:
# Save the performance table next to the notebook.
res.perf.to_csv('signals_performance.csv', index=False)
print('wrote signals_performance.csv')
res.perf[perf['sample']=='all'][['signal','horizon','mean_q','t_nw','sharpe_ann','hit']].round(4)